# Cross-Dataset Comparison

Comparative analysis of all models across all three datasets (NetFlow, KDD Cup 99, CORES-IoT).
Uses the aggregated summary from `evaluation/summary_best.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Load all evaluation results
datasets = ['netflow', 'kdd', 'cores_iot']
models = ['isolation_forest', 'sgd', 'mlp', 'ensemble']
model_labels = {'isolation_forest': 'Isolation Forest', 'sgd': 'SGD', 'mlp': 'MLP', 'ensemble': 'Ensemble'}
dataset_labels = {'netflow': 'NetFlow', 'kdd': 'KDD Cup 99', 'cores_iot': 'CORES-IoT'}

rows = []
for model in models:
    for dataset in datasets:
        try:
            eval_file = {'isolation_forest': 'isolation_forest_evaluation.csv',
                         'sgd': 'sgd_evaluation.csv',
                         'mlp': 'mlp_evaluation.csv',
                         'ensemble': 'ensemble_evaluation.csv'}[model]
            df = pd.read_csv(f'../evaluation/{model}/{dataset}/{eval_file}')
            row = df.iloc[0].to_dict()
            row['model'] = model_labels[model]
            row['dataset'] = dataset_labels[dataset]
            rows.append(row)
        except FileNotFoundError:
            print(f'Missing: {model}/{dataset}')

results = pd.DataFrame(rows)
print(f'Loaded {len(results)} results across {len(datasets)} datasets and {len(models)} models')
results[['model', 'dataset', 'accuracy', 'f1_macro']].round(4)

## 1. F1 Macro Heatmap (Stage 2 Models)

In [ ]:
# Filter Stage 2 models only
stage2 = results[results['model'] != 'Isolation Forest'].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# F1 Macro heatmap
pivot_f1 = stage2.pivot_table(index='dataset', columns='model', values='f1_macro')
pivot_f1 = pivot_f1[['SGD', 'MLP', 'Ensemble']]  # consistent order

ax = axes[0]
sns.heatmap(pivot_f1, annot=True, fmt='.4f', cmap='YlGn', ax=ax,
            vmin=pivot_f1.min().min() - 0.02, vmax=pivot_f1.max().max() + 0.005,
            linewidths=1, linecolor='white')
ax.set_title('F1 Macro by Dataset and Model', fontsize=13, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')

# Balanced Accuracy heatmap
pivot_bac = stage2.pivot_table(index='dataset', columns='model', values='balanced_accuracy')
pivot_bac = pivot_bac[['SGD', 'MLP', 'Ensemble']]

ax = axes[1]
sns.heatmap(pivot_bac, annot=True, fmt='.4f', cmap='YlGn', ax=ax,
            vmin=pivot_bac.min().min() - 0.02, vmax=pivot_bac.max().max() + 0.005,
            linewidths=1, linecolor='white')
ax.set_title('Balanced Accuracy by Dataset and Model', fontsize=13, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')

plt.tight_layout()
plt.savefig('cross_dataset_heatmaps.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Grouped Bar Chart - All Metrics

In [ ]:
colors_model = {'SGD': '#1f77b4', 'MLP': '#ff7f0e', 'Ensemble': '#2ca02c'}
dataset_names = ['NetFlow', 'KDD Cup 99', 'CORES-IoT']
model_names = ['SGD', 'MLP', 'Ensemble']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Stage 2 Model Performance Across Datasets', fontsize=15, fontweight='bold')

metric_sets = [
    ('Accuracy', 'accuracy'),
    ('F1 Macro', 'f1_macro'),
    ('Recall (Anomaly)', 'recall_anomaly'),
]

x = np.arange(len(dataset_names))
width = 0.25

for ax, (title, col) in zip(axes, metric_sets):
    for i, model in enumerate(model_names):
        model_data = stage2[stage2['model'] == model]
        values = [model_data[model_data['dataset'] == ds][col].values[0] for ds in dataset_names]
        bars = ax.bar(x + i * width, values, width, label=model,
                      color=colors_model[model], alpha=0.85, edgecolor='black', linewidth=0.5)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(dataset_names, fontsize=9)
    ax.set_ylim([0.8, 1.0])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('cross_dataset_grouped_bars.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Processing Time Comparison

In [ ]:
# Time column name differs for iso forest vs stage 2
def get_time(row):
    if 'total_time_sec' in row and pd.notna(row.get('total_time_sec')):
        return row['total_time_sec']
    return row.get('elapsed_sec', 0)

stage2['time'] = stage2.apply(get_time, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))

for i, model in enumerate(model_names):
    model_data = stage2[stage2['model'] == model]
    times = [model_data[model_data['dataset'] == ds]['time'].values[0] for ds in dataset_names]
    bars = ax.bar(x + i * width, times, width, label=model,
                  color=colors_model[model], alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}s', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Processing Time by Dataset and Model', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(dataset_names, fontsize=10)
ax.set_ylabel('Time (seconds)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('cross_dataset_times.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Stage 2 Call Rate Comparison

In [ ]:
# Stage 2 call rate is the same for all models within a dataset (determined by Isolation Forest)
call_rates = {}
for ds in dataset_names:
    ds_data = stage2[stage2['dataset'] == ds]
    call_rates[ds] = ds_data['stage2_call_rate'].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
ds_list = list(call_rates.keys())
rates = [call_rates[ds] * 100 for ds in ds_list]
colors_ds = ['#3498db', '#e74c3c', '#2ecc71']

bars = ax.barh(ds_list, rates, color=colors_ds, alpha=0.85, edgecolor='black', height=0.5)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}%', ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Stage 2 Call Rate (%)', fontsize=11, fontweight='bold')
ax.set_title('Traffic Reaching Stage 2 (by Dataset)', fontsize=13, fontweight='bold')
ax.set_xlim([0, 100])
ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('cross_dataset_call_rates.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
for ds, rate in zip(ds_list, rates):
    reduction = 100 - rate
    print(f'  {ds}: Stage 1 filters out {reduction:.1f}% of traffic (only {rate:.1f}% reaches Stage 2)')

## 5. Best Model per Dataset Summary

In [ ]:
print('=' * 90)
print('CROSS-DATASET SUMMARY - BEST MODEL PER DATASET (by F1 Macro)')
print('=' * 90)

for ds in dataset_names:
    ds_data = stage2[stage2['dataset'] == ds]
    best_idx = ds_data['f1_macro'].idxmax()
    best = ds_data.loc[best_idx]
    
    print(f'\n  {ds}:')
    print(f'    Best Model:        {best["model"]}')
    print(f'    Accuracy:          {best["accuracy"]:.4f}')
    print(f'    Balanced Accuracy: {best["balanced_accuracy"]:.4f}')
    print(f'    F1 Macro:          {best["f1_macro"]:.4f}')
    print(f'    Recall Anomaly:    {best["recall_anomaly"]:.4f}')
    time_val = best.get('total_time_sec', best.get('elapsed_sec', 0))
    print(f'    Time:              {time_val:.2f}s')

# Overall winner
model_wins = stage2.loc[stage2.groupby('dataset')['f1_macro'].idxmax()]['model'].value_counts()
print(f'\n  Overall winner: {model_wins.index[0]} (best on {model_wins.iloc[0]}/{len(dataset_names)} datasets)')
print('=' * 90)